# Visualization 02 Heatmap region

Standalone, portable notebook a name x department heat map of each decade's top names, grouped by region. Renamed from sketch 2.2. The shared setup
imports, data loading, department outlines is included
below; chart data is inlined so the HTML renders on Linux, macOS and Windows.
Running it writes the executed `_out.ipynb` and `Visualization 02 - Heatmap region.png`.

## Shared setup

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF CONTAINED and renders on
# all platforms incl. macOS: external altair data *.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches 1.13 / 2.4 / 2.6 restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory lean load: another heavy job is running on this machine ~2.5 GB free,
# so we read the CSV with categorical dtypes 15k name strings stored once, derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects the strings stay shared so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all years aggregate 2.4.
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## Name universe restriction

In [ ]:
# NOTE: no global name restriction here. 2.2 must show, per decade, THAT decade's
# top names e.g. recent decades > LÉO, GABRIEL.... Restricting to the all time
# top names would wrongly keep showing old names MARIE, JEAN in every decade.
# The inlined data still stays small because the chart cell keeps only each
# decade, sex period's top 25 names x departments see head25 below.
print(f'{named.preusuel.nunique():,} names available; per-decade top names selected in the chart.')

In [ ]:
depts = gpd.read_file('departements-version-simplifiee.geojson')
_corsica = depts['code'].isin(['2A', '2B'])
depts = pd.concat([
    depts[~_corsica],
    gpd.GeoDataFrame({'code': ['20'], 'nom': ['Corse']},
                     geometry=[depts.loc[_corsica, 'geometry'].union_all()],
                     crs=depts.crs),
], ignore_index=True)
METRO = set(depts['code'])

dept_name = named.groupby(['dpt', 'preusuel'], as_index=False)['nombre'].sum()
dept_name = dept_name.merge(dept_total, on='dpt')
dept_name['per_1000'] = (1000 * dept_name['nombre'] / dept_name['dept_total']).round(3)
dept_name = dept_name.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                            how='left').drop(columns=['code', 'dept_total'])

geo_features = alt.InlineData(values=json.loads(depts.to_json()),
                              format=alt.DataFormat(property='features'))
FRANCE_PROJECTION = dict(type='conicConformal', rotate=[-3, 0],
                         center=[0, 46.5], parallels=[44, 49])
print(f'{len(dept_name):,} (department, name) pairs embedded.')
dept_name.sample(3)

## The sketch

### Sketch 2.2 Name x department heat map, grouped by region

Each cell is the name's rate relative to its own row maximum, for the chosen decade
and sex. Departments are now grouped by region and sorted by total births
within each region, so neighbouring departments sit together; the department
code is shown on the axis full name in the tooltip and a region colour strip
with a named legend sits on top. Sliders set the decade and the number of rows; a
dropdown filters Mixed/Boys/Girls.

In [ ]:
_heat_rank_base = named.dropna(subset=['decade'])
heat_rank_mixed = (_heat_rank_base.groupby(['decade', 'preusuel'], as_index=False)['nombre'].sum()
                   .sort_values(['decade', 'nombre'], ascending=[True, False])
                   .groupby('decade').head(25))
heat_rank_mixed['sex_mode'] = 'Mixed'
heat_rank_sex = (_heat_rank_base.groupby(['decade', 'sexe', 'preusuel'], as_index=False)['nombre'].sum()
                 .sort_values(['decade', 'sexe', 'nombre'], ascending=[True, True, False])
                 .groupby(['decade', 'sexe']).head(25))
heat_rank_sex['sex_mode'] = np.where(heat_rank_sex['sexe'].eq(1), 'Boys', 'Girls')
heat_rank = pd.concat([
    heat_rank_mixed[['decade', 'sex_mode', 'preusuel', 'nombre']],
    heat_rank_sex[['decade', 'sex_mode', 'preusuel', 'nombre']],
], ignore_index=True)
heat_rank['decade'] = heat_rank['decade'].astype(int)
heat_rank['nrank'] = heat_rank.groupby(['decade', 'sex_mode'])['nombre'].rank(ascending=False, method='first').astype(int)
heat_names = heat_rank['preusuel'].drop_duplicates().tolist()
heat_rank = heat_rank[['decade', 'sex_mode', 'preusuel', 'nrank']]
hm = named[named.preusuel.isin(heat_names) & named.dpt.isin(METRO)].copy()
# Slim 4 column projection before filtering: a full frame dropna copy of the
# 3.5M row records table raised MemoryError on this RAM tight machine.
_slim = records.loc[records['decade'].notna() & records['dpt'].isin(METRO),
                    ['dpt', 'decade', 'sexe', 'nombre']]
dec_tot = (_slim.groupby(['dpt', 'decade'], as_index=False)['nombre'].sum()
           .rename(columns={'nombre': 'tot'}))
dec_tot_sex = (_slim.groupby(['dpt', 'decade', 'sexe'], as_index=False)['nombre'].sum()
           .rename(columns={'nombre': 'tot'}))
del _slim
per_dec_mixed = (hm.dropna(subset=['decade'])
           .groupby(['decade', 'dpt', 'preusuel'], as_index=False)['nombre'].sum()
           .merge(dec_tot, on=['dpt', 'decade']))
per_dec_mixed['sex_mode'] = 'Mixed'
per_dec_sex = (hm.dropna(subset=['decade'])
               .groupby(['decade', 'dpt', 'preusuel', 'sexe'], as_index=False)['nombre'].sum()
               .merge(dec_tot_sex, on=['dpt', 'decade', 'sexe']))
per_dec_sex['sex_mode'] = np.where(per_dec_sex['sexe'].eq(1), 'Boys', 'Girls')
per_dec = pd.concat([per_dec_mixed, per_dec_sex], ignore_index=True)
per_dec['per_1000'] = 1000 * per_dec['nombre'] / per_dec['tot']
heat_df = per_dec[['decade', 'dpt', 'preusuel', 'sex_mode', 'per_1000']].copy()
heat_df['rel'] = (heat_df.groupby(['decade', 'sex_mode', 'preusuel'])['per_1000']
                  .transform(lambda s: s / s.max())).round(3)
heat_df['per_1000'] = heat_df['per_1000'].round(3)
heat_df['decade'] = heat_df['decade'].astype(int)
heat_df = heat_df.merge(heat_rank, on=['decade', 'sex_mode', 'preusuel'])
heat_grid = (heat_rank.assign(_k=1)
             .merge(pd.DataFrame({'dpt': sorted(METRO), '_k': 1}), on='_k')
             .drop(columns='_k'))
heat_df = heat_grid.merge(heat_df, how='left', on=['decade', 'sex_mode', 'preusuel', 'nrank', 'dpt'])
heat_df[['per_1000', 'rel']] = heat_df[['per_1000', 'rel']].fillna(0)
heat_df = heat_df.merge(depts[['code', 'nom']], left_on='dpt', right_on='code').drop(columns='code')
# Department > metropolitan region Corsica merged as '20' 
_REGIONS = {
    'Auvergne-Rhone-Alpes': ['01', '03', '07', '15', '26', '38', '42', '43', '63', '69', '73', '74'],
    'Bourgogne-Franche-Comte': ['21', '25', '39', '58', '70', '71', '89', '90'],
    'Bretagne': ['22', '29', '35', '56'],
    'Centre-Val de Loire': ['18', '28', '36', '37', '41', '45'],
    'Corse': ['20'],
    'Grand Est': ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88'],
    'Hauts-de-France': ['02', '59', '60', '62', '80'],
    'Ile-de-France': ['75', '77', '78', '91', '92', '93', '94', '95'],
    'Normandie': ['14', '27', '50', '61', '76'],
    'Nouvelle-Aquitaine': ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87'],
    'Occitanie': ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82'],
    'Pays de la Loire': ['44', '49', '53', '72', '85'],
    'Provence-Alpes-Cote d Azur': ['04', '05', '06', '13', '83', '84'],
}
REGION_OF = {d: r for r, ds in _REGIONS.items() for d in ds}
assert set(METRO) <= set(REGION_OF), set(METRO) - set(REGION_OF)
heat_df['region'] = heat_df['dpt'].map(REGION_OF)

# Columns grouped by region, then by total births within the region, so adjacent
# departments are related; a region colour strip names the groups above the grid.
dept_meta = (dept_total[dept_total['dpt'].isin(METRO)]
             .merge(depts[['code', 'nom']], left_on='dpt', right_on='code').drop(columns='code'))
dept_meta['region'] = dept_meta['dpt'].map(REGION_OF)
dept_meta = dept_meta.sort_values(['region', 'dept_total'], ascending=[True, False])
dept_order = dept_meta['dpt'].tolist()

n_rows = alt.param(name='n_rows', value=15,
                   bind=alt.binding_range(min=5, max=25, step=1, name='Top N names rows '))
decade_22 = alt.param(name='decade_22', value=2010,
                      bind=alt.binding_range(min=1900, max=2020, step=10, name='Decade '))
sex_22 = alt.param(name='sex_22', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS, name='Sex '))

# Explicit region palette, shared by the strip AND the heatmap cells so each cell
# is a monochrome shade of ITS region's colour intensity = rate vs row max.
REGION_COLORS = {
    'Auvergne-Rhone-Alpes': '#4e79a7', 'Bourgogne-Franche-Comte': '#a0cbe8',
    'Bretagne': '#f28e2b', 'Centre-Val de Loire': '#ffbe7d', 'Corse': '#59a14f',
    'Grand Est': '#8cd17d', 'Hauts-de-France': '#b6992d', 'Ile-de-France': '#f1ce63',
    'Normandie': '#499894', 'Nouvelle-Aquitaine': '#86bcb6', 'Occitanie': '#e15759',
    'Pays de la Loire': '#ff9d9a', 'Provence-Alpes-Cote d Azur': '#79706e',
}
region_scale = alt.Scale(domain=list(REGION_COLORS), range=list(REGION_COLORS.values()))

region_strip = alt.Chart(dept_meta).mark_rect(stroke='white', strokeWidth=0.3).encode(
    x=alt.X('dpt:N', sort=dept_order, axis=None),
    color=alt.Color('region:N', title='Region', scale=region_scale),
    tooltip=[alt.Tooltip('nom:N', title='Department'), alt.Tooltip('dpt:N', title='Code'),
             alt.Tooltip('region:N', title='Region')],
).properties(width=820, height=16, title='Region')

heat = alt.Chart(heat_df).transform_filter(
    'datum.decade == decade_22 && datum.sex_mode == sex_22 && datum.nrank <= n_rows'
).mark_rect().encode(
    x=alt.X('dpt:N', title='Department grouped by region; code shown, name in tooltip',
            sort=dept_order, axis=alt.Axis(labelAngle=-90, labelFontSize=7, ticks=False)),
    y=alt.Y('preusuel:N', title='Name',
            sort=alt.EncodingSortField(field='nrank', op='min', order='ascending')),
    color=alt.Color('region:N', scale=region_scale, legend=None),
    opacity=alt.Opacity('rel:Q', scale=alt.Scale(domain=[0, 1], range=[0.06, 1]),
                        legend=alt.Legend(title=['Cell intensity', 'vs row max'], format='.0%')),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('decade:O', title='Decade'), alt.Tooltip('nom:N', title='Department'),
             alt.Tooltip('region:N', title='Region'),
             alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f'),
             alt.Tooltip('rel:Q', title='vs row max', format='.0%')],
).properties(width=820, height=360)

sketch_2_2 = alt.vconcat(region_strip, heat, spacing=3).add_params(
    decade_22, sex_22, n_rows).resolve_scale(color='independent').properties(
    title='Visualization 02 Regional profile of the selected decade top names')

sketch_2_2.save('Visualization 02 - Heatmap region.png', ppi=120)
sketch_2_2
